# Dynamic Functional Connectivity (DFC)

---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO/blob/main/Borriero_4_dynamic_functional_connectivity_SCHAEFER.ipynb)

## Importing

In [ ]:
! pip install nibabel nilearn nitime statsmodels seaborn pycirclize networkx

In [ ]:
import nilearn as nil
import nibabel as nib
import numpy as np
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
from nilearn import plotting, surface, datasets, input_data, decomposition
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm # For having progressbar during loops
from nitime.timeseries import TimeSeries
from nitime.analysis import FilterAnalyzer
from sklearn.decomposition import PCA
from nilearn.connectome import ConnectivityMeasure
from matplotlib import colors
from matplotlib import patches
from pycirclize import Circos
import sys
import networkx as nx
import community as community_louvain
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score
from itertools import combinations
from statannotations.Annotator import Annotator




# Download some files

In [ ]:
# A wrapper method to download files

import os, subprocess, shlex
import os
import subprocess

def download_from_drive(file_keyword, base_folder="."):
    file_name, file_ID = files_dict[file_keyword]
    DEST_PATH = os.path.join(base_folder, file_name)
    if not os.path.exists(DEST_PATH):
        URL = f"https://drive.usercontent.google.com/download?id={file_ID}&confirm=t"
        cmd = ["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
            "--no-check-certificate", "-O", DEST_PATH, URL]
        ret = subprocess.call(cmd)  # shell=False by default
        if ret != 0:
            raise RuntimeError("Download failed. Check sharing settings or FILE_ID.")
        else:
            print("Downloaded " + file_name + "!")
    else:
        print(file_name + " already present:", DEST_PATH)

In [ ]:
files_dict = {'glasser_labels' : ('glasser_labels.csv','12LumqrRkp1rUAJGg8h3tOIwWubhyyoqY'),
              'glasser_atlas':   ('glasser_atlas.nii.gz','1GeB5pi25SRnyit8s5siDUCFFmJfsB3Yw'),
              'schaefer_labels': ('schaefer_labels.csv', '1Lgh4X9hud2xXGxYEWADFxfMe81P_tTfS'),
              'schaefer_atlas':  ('schaefer_atlas.nii.gz', '1acSTf6m9TyJxm2ruTcwd9eC8PNGfG1PX'),
              'subj_01_rest':  ('subj_01_rest.nii.gz', '1XXhVrMymlaBE0ekbgbHoH8fVYteWFGIa'),
              'subj_03_rest':  ('subj_03_rest.nii.gz', '1RziRBrPUFemlHfjPMoeJX26CbHhqlKM4'),
              'subj_04_rest':  ('subj_04_rest.nii.gz', '10thmeWWgk31FaBk0Qi0n_rtOef5YVAjM'),
              'subj_05_rest':  ('subj_05_rest.nii.gz', '1-_ifL8yQWAoUiU7v7niz6O9YbVGyj4xn'),
              'subj_06_rest':  ('subj_06_rest.nii.gz', '1iE-5pfybf8imraDfJvP6A-XuXIyHVHQD'),
              'subj_01_sleep':  ('subj_01_sleep.nii.gz', '1zAU1sI9jS0wzGp_raNUemxUYPilLIZJH'),
              'subj_03_sleep':  ('subj_03_sleep.nii.gz', '1C9pPi7PwTcdg4Nfz1xVVCFToghRTxSQQ'),
              'subj_04_sleep':  ('subj_04_sleep.nii.gz', '1fn6xCluS7lEasPKgHafThwvhA11Mo0A5'),
              'subj_05_sleep':  ('subj_05_sleep.nii.gz', '1DkWrsAa3-jLtVggLIujtxNfqhgwezhea'),
              'subj_06_sleep':  ('subj_06_sleep.nii.gz', '1PvG2KQMF40QnmNs6Z5kMx8abKk_tBH3C'),
              'all_subj_mask':  ('all_subj_mask.nii.gz', '1azzh7yXC4uG8clHv4Zm7sBZgta0dTMDf')}

In [ ]:
for key in files_dict.keys():
  download_from_drive(key)

# Utils

In [ ]:
def signal_filter(nifti_img, mask, standardize=False, ub=0.08, lb=0.008):
    print('Filtering the signal...')
    img = nifti_img.get_fdata()
    img_affine = nifti_img.affine
    TR = nifti_img.header.get_zooms()[3]
    
    n_vox_mask = np.sum(mask==1) # Number of voxels in the mask
    mask_cord = np.where(mask==1)
    img_filtered = np.zeros(img.shape)
#     print(img_filtered.shape)
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = img[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        T = TimeSeries(voxel_ts.T, sampling_interval=TR)
        F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
        tmp_TC_filt = F.filtered_boxcar.data
        tmp_TC_filt = tmp_TC_filt.T
        if standardize:
            tmp_TC_filt = (tmp_TC_filt-np.mean(tmp_TC_filt))/np.std(tmp_TC_filt)
            
        # Check if there are nan values
        if np.sum(np.isnan(tmp_TC_filt))!=0:
            for t in tmp_TC_filt:
                tmp_TC_filt[t] = 0
        img_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

    return nib.Nifti1Image(img_filtered, img_affine)

In [ ]:
def makeboxplot_1group(dataframe, my_pal, ylabel, title='', ylim=[-2,2], annot_test='t-test_ind'):
    keys = [key for key in dataframe.keys()]
    groups_order =  np.unique(dataframe[keys[1]]).tolist()
    pairs = list(combinations(groups_order, 2))
    hue_plot_params = {
        'data': dataframe,
        'x': keys[1],
        'y': keys[0],
        "order": groups_order,
        "palette": my_pal
    }

    
    fig,ax = plt.subplots(1,1, figsize=(4.5,3.5))
    with sns.plotting_context("notebook", font_scale = 1.2):
        sns.violinplot(**hue_plot_params, ax=ax, width=0.6, inner="quart", linewidth=2)
        annotator = Annotator(ax, pairs, **hue_plot_params)
        annotator.configure(test=annot_test, text_format='star', loc='outside')
                            # ,comparisons_correction='Benjamini-Hochberg')
        _, stat = annotator.apply_and_annotate()
        # plt.title(ylabel)
        plt.tight_layout()
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_ylim(ylim[0],ylim[1])
    ax.set_title(title,y=1.2)

In [ ]:
def makeboxplot(dataframe, palette, ylabel, title, annot_test='t-test_ind', figsize=(10.5,5)):
    """
    Dataframe in a format: Data, subgroup (e.g network), group
    """
    keys = [key for key in dataframe.keys()]
    groups_order =  np.unique(dataframe[keys[2]]).tolist()
    subgroups = np.unique(dataframe[keys[1]]).tolist()

    group_pairs = list(combinations(groups_order, 2))
    
    # then for each subgroup and each group_pair, make your nested pair
    pairs = [
            [(sub, g1), (sub, g2)]
            for g1, g2 in group_pairs
            for sub in subgroups
            ]
    
    #     annot_test =  'Brunner-Munzel'
    hue_plot_params = {
        'data': dataframe,
        'x': keys[1],
        'y': keys[0],
        "order": subgroups,
        "hue": keys[2],
        "hue_order": groups_order,
        "palette": palette
    }

    # hue_plot_params = {
    #     'data': dataframe,
    #     'x': 'network',
    #     'y': 'data',
    #     "order": subgroups,
    #     "hue": 'group',
    #     "hue_order": groups_order,
    #     "palette": palette
    # }
    
    fig,ax = plt.subplots(1,1, figsize=figsize)
    with sns.plotting_context("notebook", font_scale = 1.2):
        # Create new plot
    #     ax = get_log_ax()
    
        # Plot with seaborn
        sns.boxplot(ax=ax, **hue_plot_params, showfliers=False, width=0.5, gap=.1,
                    boxprops     = dict(linewidth=2.5),
                    whiskerprops = dict(linewidth=2.5),
                    capprops     = dict(linewidth=2.5),
                    medianprops  = dict(linewidth=2.5),
                    flierprops   = dict(markeredgewidth=1))
        # sns.violinplot(**hue_plot_params, ax=ax, width=0.6, inner="quart", linewidth=2)

        ax.set_xlabel('group', fontsize=20)
        ax.set_ylabel(ylabel, fontsize=20)
    
        # Add annotations
        annotator = Annotator(ax, pairs, **hue_plot_params)
        annotator.configure(test=annot_test, text_format='star', loc='inside',comparisons_correction='Benjamini-Hochberg')
        annotator._pvalue_format.pvalue_thresholds =  [[0.0001, '****'], [0.001, '***'], [0.01, '**'], [0.05, '*'], [1, 'ns']]
        _, stat = annotator.apply_and_annotate()
    
        # Label and show
        plt.legend(loc='lower left', fontsize=10,) #bbox_to_anchor=(1.0,1.0)
        plt.title(title, fontsize=16)
    plt.tight_layout()


## Load an example subject

In [ ]:
main_path = '/content/'
file_name = 'subj_01_rest.nii.gz'

sub_ = nib.load(main_path+file_name) # Load the nii.gz file of the subject
sub_header = sub_.header # Subject's header
sub_affine = sub_.affine # Subject's affine
sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
print(sub.shape) # x*y*z*time

TR = sub_.header.get_zooms()[3]
time_cutoff = 286 # To cut the sleep run

# Load a brain mask

In [ ]:
main_path = '/content/'
file_name = 'all_subj_mask.nii.gz' 
mask_ = nib.load(main_path+file_name)
mask = mask_.get_fdata()

# Load the atlas

In [ ]:
main_path = '/content/'
file_name = 'schaefer_atlas.nii.gz'
file_name_labels = 'schaefer_labels.csv'
path = main_path+file_name
path_lab = main_path+file_name_labels

atlas_ = nib.load(path) # Load the nii.gz file of the subject
atlas_header = atlas_.header # Subject's header
atlas_affine = atlas_.affine # Subject's affine
atlas = atlas_.get_fdata() # Get the numpy version of the nii.gz file
print(atlas.shape) # x*y*z*time

# Labels
atlas_labels = pd.read_csv(path_lab, delimiter=';')
print(atlas.min(), atlas.max())

# Define a mask based on the atlas
atlas_mask = np.zeros(atlas.shape)
atlas_mask[np.where(atlas!=0)] = 1
atlas_mask_ = nib.Nifti1Image(atlas_mask, atlas_affine)

# DFC step 1: define the functional connectivity over sliding windows and concatenate all the subjects

In [ ]:
# # With phase locking and hilbert tansform
# import numpy as np
# import nibabel as nib
# from scipy.signal import hilbert
# from tqdm import tqdm

# sub_list = ['01','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','22','23','24','25','26','27','28','29','30','31','32','33']
# sub_list = ['01','03','04','05','06','07','08','09','10','11']
# sub_list = ['01','03']


# main_path = 'E:/Sleep/'

# # Initialize global container
# ipl_all_sub = np.zeros((0, int(atlas.max()), int(atlas.max())))

# for rest_or_sleep in ['rest', 'sleep']:
#     print(f"=== {rest_or_sleep.upper()} ===")

#     for sub_i, sub in enumerate(sub_list):
#         file_name = 'sub-'+sub+'_task-'+rest_or_sleep+'_run-1_bold_STD_ants_FIX.nii.gz'
#         print(f"Subject {sub_i+1}: {file_name}")

#         # --- Load and filter ---
#         sub_img = nib.load(main_path + file_name)
#         sub_filtered_img = signal_filter(sub_img, atlas_mask)  # user-defined filtering function
#         sub_filtered = sub_filtered_img.get_fdata()  # (x, y, z, time)
#         n_timepoints = sub_filtered.shape[-1]

#         # --- Parcellate ---
#         n_rois = int(atlas.max())
#         sub_parc = np.zeros((n_rois, n_timepoints))

#         for a_idx in range(1, n_rois + 1):
#             area_coord = np.where(atlas == a_idx)
#             area_ts = sub_filtered[area_coord].mean(axis=0)
#             sub_parc[a_idx - 1, :] = area_ts

#         # --- Compute phase using Hilbert transform ---
#         analytic_signal = hilbert(sub_parc, axis=1)
#         phase = np.angle(analytic_signal)  # shape (n_rois, time)

#         # --- Compute instantaneous phase coherence (vectorized) ---
#         # Complex phase representation: exp(i * phase)
#         complex_phase = np.exp(1j * phase)  # (n_rois, time)
#         n_time = complex_phase.shape[1]

#         # Compute outer product at each time point: exp(i*(phi_i - phi_j))
#         ipl_t = np.einsum('it,jt->tij', complex_phase, np.conjugate(complex_phase))
#         ipl_t = np.abs(ipl_t)  # magnitude gives instantaneous phase coherence (0-1)
#         np.fill_diagonal(ipl_t[0], 0)  # set diagonal to zero (self connections)

#         # --- Concatenate across subjects ---
#         ipl_all_sub = np.concatenate((ipl_all_sub, ipl_sub), axis=0)

#         print("  Current total IPL shape:", ipl_all_sub.shape)

In [ ]:
%%time
# We have to chose the length and the stride of the sliding windows
win_size = 20 # the TR is the time unit. In our case the TR is equal to 1s   --- 6
stride = 3  # --- 1

# Define the correlation measures
correlation_measure = ConnectivityMeasure(kind='correlation')

# sub_list = ['01','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','22','23','24','25','26','27','28','29','30','31','32','33']
# sub_list = ['01','03','04','05','06','07','08','09','10','11']
sub_list = ['01','03','04','05','06']


# sub_list = ['01','03','04']


main_path = '/content/'
corr_matrix_all_sub = np.zeros((0,int(atlas.max()), int(atlas.max()))) # Here we concatenate all the fc matrices over time of all the subjects
for rest_or_sleep in ['rest','sleep']:
    all_sub_parc = [] # Concatenate here all the subjects
    for sub_i,sub in enumerate(sub_list): # Loop over the subjects    
        file_name = 'sub_'+sub+'_'+rest_or_sleep+'.nii.gz'
        print('Subject',str(sub_i+1))
        sub_ = nib.load(main_path+file_name)
        sub_filtered_ = signal_filter(sub_, atlas_mask)
        sub_filtered = sub_filtered_.get_fdata()[:,:,:,:time_cutoff]
        sub_parc = np.zeros((0,sub_filtered.shape[-1])) # parcellized brain
        for a_idx in range(1, int(atlas.max())+1): # Loop over areas
            area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
            area_ts = sub_filtered[area_coord].mean(axis=0) # time series of the current region
            sub_parc = np.concatenate((sub_parc, area_ts[np.newaxis]), axis=0)
        # Loop over the entire time series
        t=0 
        while t+win_size<sub_filtered.shape[-1]: 
            # print(sub_parc[:,t:t+win_size].shape)
            correlation_matrix = correlation_measure.fit_transform([sub_parc[:,t:t+win_size].T])
            corr_matrix_all_sub = np.concatenate((corr_matrix_all_sub, correlation_matrix), axis=0)
            t += stride 
            # print(corr_matrix_all_sub.shape)
        
        print(corr_matrix_all_sub.shape)

In [ ]:
# ist_connectivity = ipl_all_sub 

In [ ]:
# Plot a few of the connectivity matrix over time for a subject
plt.imshow(corr_matrix_all_sub[110], cmap='PiYG')
plt.xticks([]);
plt.yticks([]);
# plt.savefig('ex_mat_110.png', dpi=300)

# DFC step 2: clusterize each istantaneous connectivity

## Version 1: clusterize the full connectivity matrices

### Let's start choosing a fixed value for the number of cluster 

In [ ]:
%%time
# Flatten all the connectivity matrices
corr_matrix_all_sub_flattened = np.reshape(corr_matrix_all_sub, (corr_matrix_all_sub.shape[0], int(atlas.max())*int(atlas.max())))
print(corr_matrix_all_sub_flattened.shape)

# Apply Kmeans
K = 4
kmeans = KMeans(n_clusters=K, random_state=0)  # Specify the number of clusters
kmeans.fit(corr_matrix_all_sub_flattened)

# Get the cluster labels and centers
labels = kmeans.labels_       # Cluster assignments for each matrix
centers = kmeans.cluster_centers_  # Cluster centers in flattened form

print(labels.shape, centers.shape)
centers = np.reshape(centers, (K, int(atlas.max()),int(atlas.max())))
print(centers.shape)

In [ ]:
# Plot the dyanmics of the subjects within the set of states
sub_labels = labels[:116]
fig, ax = plt.subplots(1,)
ax.plot(sub_labels, linewidth=1.5)
ax.set_yticks([k for k in range(4)], labels=[k for k in range(1,5)])
ax.set_xlabel('time');
ax.set_ylabel('states');

In [ ]:
# Plot the centroids
fig, axs = plt.subplots(1,K, figsize=(15, 3))
for i,ax in enumerate(axs.flat):
    sns.heatmap(centers[i], ax=ax, square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title('Centroid '+str(i+1))

Not so easy to read...

## Version 2: clusterize the leading eigenvector of each istantaneous connectivity matrix

In [ ]:
# Function to extract leading eigenvectors
def leading_eigenvectors(matrices, n_leading=1):
    leading_eigenvecs = []
    
    for matrix in tqdm(matrices):
        # Step 1: Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eig(matrix)
        
        # Step 2: Sort eigenvalues in descending order and select the top ones
        idx = np.argsort(eigenvalues)[::-1]  # Indices for sorting eigenvalues in descending order
        leading_vectors = eigenvectors[:, idx[:n_leading]]  # Select top n_leading eigenvectors
        
        #### Change the sign of the leading eigenvector if necessary
        if np.sum(leading_vectors)>0:
            leading_vectors=-leading_vectors;

        ### Show to the students the effect of this trick
        
        leading_eigenvecs.append(leading_vectors)
    
    return leading_eigenvecs

In [ ]:
# Extract the leading eigenvpectors from leading connecitivity matrix
n_leading = 1  # Number of leading eigenvectors to extract
eigenvectors = leading_eigenvectors(corr_matrix_all_sub, n_leading)
eigenvectors = [v.real[:,0] for v in eigenvectors]
len(eigenvectors)

In [ ]:
fig = plt.figure(figsize=(1,5))
plt.title('eigenvector example')
sns.heatmap(eigenvectors[80][:,np.newaxis], cbar=False, cmap='magma')
plt.xticks([])
plt.yticks([])
plt.savefig('vector_ex_80', dpi=300)

In [ ]:
%%time
# Apply the kmeans to the set of leading eigenvectors
# Apply Kmeans
K = 4
kmeans = KMeans(n_clusters=K, random_state=0)  # Specify the number of clusters
kmeans.fit(eigenvectors)

# Get the cluster labels and centers
labels = kmeans.labels_       # Cluster assignments for each matrix
centers = kmeans.cluster_centers_  # Cluster centers in flattened form
print(labels.shape, centers.shape)

In [ ]:
# Plot the eigenvectors
fig, axs = plt.subplots(1, K, figsize=(8,4))
for i,ax in enumerate(axs.flat):
    ax.set_title('STATE {}' .format(i+1), fontsize=15)
    # ax.barh(range(centroids[k_].shape[0]), -centroids[k_], height=1);
    # #              tick_label=nodes_labels)
    ax.barh(range(centers[i].shape[0]), centers[i], height=1, color='green');
    # ax.barh(range(centroids[i].shape[0]), centroids[i], height=1, color='purple');
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel('ROIs', fontsize=15)
    fig.tight_layout() 

In [ ]:
# Plot the centroids over the brain
for c_i,center in enumerate(centers):
    vmin=center.min()
    vmax=center.max()
    brain_centroid = np.zeros((atlas.shape))
    for area_i, area_c in enumerate(center):
        area_coord = np.where(atlas==area_i+1)
        brain_centroid[area_coord[0], area_coord[1], area_coord[2]] = area_c
    nii_img = nib.Nifti1Image(brain_centroid, atlas_affine)

    # Plot the betweenness centrality over the brain
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan 

    cmap='RdBu_r'
    fig, axs = plt.subplots(2,2,subplot_kw={'projection': '3d'})
    # left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                title='Left lateral', colorbar=True, axes=axs[0][0], cmap=cmap,
                                vmin=-vmax, vmax=vmax)
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                title='Left medial', colorbar=True, view='medial', axes=axs[1][0], cmap=cmap,
                                vmin=-vmax, vmax=vmax)
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                title='Right lateral', colorbar=True, view='lateral', axes=axs[0][1], cmap=cmap,
                                vmin=-vmax, vmax=vmax)
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                title='Right medial', colorbar=True, view='medial', axes=axs[1][1], cmap=cmap,
                                vmin=-vmax, vmax=vmax)
    plt.suptitle('State '+str(c_i+1))

    # Show the plot
    plotting.show()

In [ ]:
fig, axs = plt.subplots(1,K, subplot_kw={'projection': '3d'})
for c_i, ax in enumerate(axs.flat):
    center = centers[c_i] 
    vmin=center.min()
    vmax=center.max()
    brain_centroid = np.zeros((atlas.shape))
    for area_i, area_c in enumerate(center):
        area_coord = np.where(atlas==area_i+1)
        brain_centroid[area_coord[0], area_coord[1], area_coord[2]] = area_c
    nii_img = nib.Nifti1Image(brain_centroid, atlas_affine)
    
    # Plot the betweenness centrality over the brain
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 

    cmap='RdBu_r'
    # left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                title='Left lateral', colorbar=False, view='lateral', axes=ax, cmap=cmap,
                                vmin=-vmax, vmax=vmax)
    ax.set_title('State '+str(c_i+1))
plt.suptitle('K='+str(K),y=0.7);

## Try to find the optimal number of cluster using the silhouette score

In [ ]:
%%time
# Loop over a range of posssible values of K
silhouette_scores = []
davis_bouldin_scores = []
for K in range(3, 20):
    kmeans = KMeans(n_clusters=K, random_state=0)  # Specify the number of clusters
    kmeans.fit(eigenvectors)
    
    # Get the cluster labels and centers
    labels = kmeans.labels_       # Cluster assignments for each matrix
    centers = kmeans.cluster_centers_  # Cluster centers in flattened form

    silhouette_scores.append(silhouette_score(eigenvectors, labels))
    davis_bouldin_scores.append(davies_bouldin_score(eigenvectors, labels))

In [ ]:
cmap = matplotlib.colormaps['Set2']
colors = cmap(np.linspace(0, 1, 8))
fig, ax = plt.subplots(1,)
ax.plot(silhouette_scores, linewidth=1.5, color=colors[0])
ax2 = ax.twinx()
ax2.plot(davis_bouldin_scores, linewidth=1.5, color=colors[1])
ax.set_xticks([x for x in range(0,20-3)], labels=[x for x in range(3,20)]);
ax.set_ylabel('Silhouette score', color=colors[0], fontsize=15)
ax2.set_ylabel('Davis Bouldin score', color=colors[1], fontsize=15)
ax.set_xlabel('K')

### Plot the brains associated to the states at different K (to check the robustness of the method)

In [ ]:
# Loop over a range of posssible values of K
silhouette_scores = []
davis_bouldin_scores = []
for K in range(3, 9):
    kmeans = KMeans(n_clusters=K, random_state=0)  # Specify the number of clusters
    kmeans.fit(eigenvectors)
    
    # Get the cluster labels and centers
    labels = kmeans.labels_       # Cluster assignments for each matrix
    centers = kmeans.cluster_centers_  # Cluster centers in flattened form

    silhouette_scores.append(silhouette_score(eigenvectors, labels))
    davis_bouldin_scores.append(davies_bouldin_score(eigenvectors, labels))
    fig, axs = plt.subplots(1,K, subplot_kw={'projection': '3d'})
    for c_i, ax in enumerate(axs.flat):
        center = centers[c_i] 
        vmin=center.min()
        vmax=center.max()
        brain_centroid = np.zeros((atlas.shape))
        for area_i, area_c in enumerate(center):
            area_coord = np.where(atlas==area_i+1)
            brain_centroid[area_coord[0], area_coord[1], area_coord[2]] = area_c
        nii_img = nib.Nifti1Image(brain_centroid, atlas_affine)
        
        # Plot the betweenness centrality over the brain
        fsaverage = datasets.fetch_surf_fsaverage()
        texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
        texture_left[np.where(texture_left==0)] = np.nan 
    
        cmap='RdBu_r'
        # left hemisphere
        plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                    title='Left lateral', colorbar=False, view='lateral', axes=ax, cmap=cmap,
                                    vmin=-vmax, vmax=vmax)
        ax.set_title('State '+str(c_i+1), fontsize=8)
    plt.suptitle('K='+str(K),y=0.75)

### Measure the lifetime and the percentage of occurence of each state

In [ ]:
# Apply the kmeans to the set of leading eigenvectors
# Apply Kmeans
K = 4
kmeans = KMeans(n_clusters=K, random_state=0)  # Specify the number of clusters
kmeans.fit(eigenvectors)

# Get the cluster labels and centers
labels = kmeans.labels_       # Cluster assignments for each matrix
centers = kmeans.cluster_centers_  # Cluster centers in flattened form
print(labels.shape, centers.shape)

In [ ]:
np.sum(labels[:286]==1)/time_points

In [ ]:
np.sum(labels[286:2*286]==1)/time_points

In [ ]:
len(labels)/20

In [ ]:
# We want to measure the life time and the percentage of occupancy of each state for each subject, in order to make a statistic

# np.where(ts==s)[0].shape[0]/time_points
n_sub = len(sub_list)
time_points = int(len(eigenvectors)/(n_sub*2)) # number of sliding windows for each subject
# Percentage of occupancy
PO_vals = []
condition_list = []
state_list = []
for k in range(K): # Loop over states
    sub_count = 0
    for rest_or_sleep in ['rest','sleep']:
        for sub_i in range(n_sub): # Loop over subject
            sub_eigvecs = eigenvectors[sub_count*time_points:(sub_count+1)*time_points]
            sub_labels = labels[sub_count*time_points:(sub_count+1)*time_points]

            # Percentage of occupancy of the current state
            count = np.sum(sub_labels==k) # How many time the subject goes in the state k 
            PO = count/time_points
            PO_vals.append(PO)
            condition_list.append(rest_or_sleep)
            state_list.append(str(k+1))

            sub_count += 1
print(sub_count)
PO_df = pd.DataFrame({'PO':PO_vals, 'state':state_list, 'condition':condition_list})

# Lifetime
LT_vals = []
condition_list = []
state_list = []
sub_count = 0
for rest_or_sleep in ['rest','sleep']:
    for sub_i in range(n_sub): # Loop over subject
        print(sub_count)
        sub_eigvecs = eigenvectors[sub_count*time_points:(sub_count+1)*time_points]
        sub_labels = labels[sub_count*time_points:(sub_count+1)*time_points]
        
        # Lifetime of the current state
        sub_LT_dict = {} # dict containing length of each segment of each state
        for k in range(K):
            sub_LT_dict[k] = []
        curr_value = sub_labels[0]
        curr_length = 1
        for i in range(len(sub_labels)):
            if sub_labels[i]==curr_value:
                curr_length += 1
            else:
                sub_LT_dict[curr_value].append(curr_length)
                curr_value = sub_labels[i]
                curr_length = 1
        # Save the results for the current subject
        for k in range(K):
            LT_vals.append(np.mean(sub_LT_dict[k]))
            condition_list.append(rest_or_sleep)
            state_list.append(str(k+1))

        sub_count +=1
LT_df = pd.DataFrame({'LT':LT_vals, 'state':state_list, 'condition':condition_list})


In [ ]:
palette = {'sleep':'salmon', 'rest':'cadetblue'}
makeboxplot(PO_df, palette, 'PO', '', annot_test='t-test_ind', figsize=(10.5,5))
makeboxplot(LT_df, palette, 'LT', '', annot_test='t-test_ind', figsize=(10.5,5))

### Transition probability
Compute the probability to go from a state to another state

In [ ]:
rest_or_sleep_matrices = []
sub_count = 0
for rest_or_sleep in ['rest','sleep']:
    # Initialize the transition probability matrix
    all_sub_trans_prob_matrix = np.zeros((n_sub,K,K))
    for sub_i in range(n_sub): # Loop over subject
        sub_eigvecs = eigenvectors[sub_count*time_points:(sub_count+1)*time_points]
        sub_labels = labels[sub_count*time_points:(sub_count+1)*time_points]

        prev_value = sub_labels[0]
        for i in range(len(sub_labels)):
            curr_value = sub_labels[i]
            all_sub_trans_prob_matrix[sub_i, prev_value, curr_value] += 1
            prev_value = curr_value
        # Normalize each row for the total number of transition from that state
        for k in range(K):
            total_trans = np.sum(np.sum(all_sub_trans_prob_matrix[sub_i, k, :]))
            if total_trans == 0:
                all_sub_trans_prob_matrix[sub_i, k, :] = 0
            else:
                all_sub_trans_prob_matrix[sub_i, k, :] /= total_trans
        sub_count += 1
    rest_or_sleep_matrices.append(all_sub_trans_prob_matrix)

            

In [ ]:
t[0]

In [ ]:
from scipy.stats import ttest_ind, false_discovery_control
K = 4
t_matrix = np.zeros((K, K))
p_matrix = np.zeros((K, K))
p_matrix_corrected = np.zeros((K, K))

all_pvals = []
for n_row in enumerate(range(K)):
    for n_col in enumerate(range(K)):
        t, p = ttest_ind(list(rest_or_sleep_matrices[0][:,n_row,n_col]), list(rest_or_sleep_matrices[1][:,n_row,n_col]))
        t_matrix[n_row,n_col] = t[0]
        p_matrix[n_row,n_col] = p[0]
        all_pvals.append(p[0])
all_pvals_corrected = false_discovery_control(all_pvals)
count = 0
for n_row in range(K):
    for n_col in range(K):
        p_matrix_corrected[n_row,n_col] = all_pvals_corrected[count]
        count +=1   

In [ ]:
state_list = [x+1 for x in range(4)]
fig, axs = plt.subplots(1,3, figsize=(18,5))
sns.heatmap(t_matrix, annot=True, xticklabels=state_list, yticklabels=state_list, ax=axs[0], square=True)
sns.heatmap(p_matrix, annot=True, xticklabels=state_list, yticklabels=state_list, ax=axs[1], square=True)
sns.heatmap(p_matrix_corrected, annot=True, xticklabels=state_list, yticklabels=state_list, ax=axs[2], square=True)